## T-graph maker
**For determining transition pathways, one by one, in a system of N coupled hysterons.**

**Joseph D. Paulsen, November 24, 2025**

paulse8@stolaf.edu

-----------

You have N hysterons with known transition thresholds Hoff and Hon and an interaction matrix J. The hysterons are subjected to a global field H. Given a system state, S, and field value, the following functions can determine:
* the next stable state upon increasing field;
* the next stable state upon decreasing field;
* the range of global field values where the state is stable (if any).

This software was used to flesh out the T=2 transtion graph for four coupled hysterons in "Mechanical hysterons with tunable interactions of general sign," Joseph D. Paulsen, arXiv 2409.07726 (2024), available at https://doi.org/10.48550/arXiv.2409.07726.

Several key functions were adapted from “hysteron 0.1” by Nathan C. Keim and Joseph D. Paulsen, https://github.com/nkeim/hysteron (2020). The hysteron parameters for this T=2 cycle were discovered via a random search by Nathan C. Keim using “hysteron 0.1”. 

In [ ]:
import numpy as np

In [ ]:
def threshu(Hon,J,S): # values of gamma_i^+(S), but infinity if s_i==1.
    vals = Hon - J @ S
    for i in range(len(S)):
        if S[i] == 1:
            vals[i] = np.inf
    return vals
    
def threshd(Hoff,J,S): # values of gamma_i^-(S), but -infinity if s_i==-1.
    vals = Hoff - J @ S
    for i in range(len(S)):
        if S[i] == -1:
            vals[i] = -np.inf
    return vals

def avalanche(Hon,Hoff,J,S,H): # evolve the most unstable hysteron, if any, at the present field
    dist = 0
    lookup = max(H - threshu(Hon,J,S))
    lookdn = min(H - threshd(Hoff,J,S))
    if lookup > 0:     # most wanting to flip up (if any)
        dist = lookup
        i = np.argmax(H - threshu(Hon,J,S))
    if lookdn < -dist: # most wanting to flip down (if any)
        dist = lookdn
        i = np.argmin(H - threshd(Hoff,J,S))
    if dist != 0:
        S[i] *= -1         # flip it!
    return S

def settle(Hon,Hoff,J,S,H): # settle to a stable state
    while (H != np.inf) and (H != -np.inf):
        Sold = np.copy(S)
        S = avalanche(Hon,Hoff,J,S,H)
        if np.array_equal(S,Sold):
            return S
    return S

def u(Hon,J,S,H): # evolve the system "up" and stop at next stable state
    if min(S) == 1:
        return H
    else:
        if len(np.where(threshu(Hon,J,S) == threshu(Hon,J,S).min())[0]) > 1:
            print("WARNING: degenerate! Flipping lowest i.")
        i = np.argmin(threshu(Hon,J,S))
        Hval = min(threshu(Hon,J,S))
        S[i] = 1.
        if Hval > H: # so as to not revert the field
            H = Hval
        S = settle(Hon,Hoff,J,S,H)
        return H

def d(Hoff,J,S,H): # evolve the system "down" and stop at next stable state
    if max(S) == -1:
        return H
    else:
        if len(np.where(threshd(Hoff,J,S) == threshd(Hoff,J,S).max())[0]) > 1:
            print("WARNING: degenerate! Flipping lowest i.")
        i = np.argmax(threshd(Hoff,J,S))
        Hval = max(threshd(Hoff,J,S))
        S[i] = -1.
        if Hval < H: # so as to not revert the field
            H = Hval
        S = settle(Hon,Hoff,J,S,H)
        return H

def stablerange(Hon,Hoff,J,S): # returns interval of H where state S is stable
    up = min(threshu(Hon,J,S))
    dn = max(threshd(Hon,J,S))
    if up < dn:
        print("no stable range")
        return
    else:
        print(dn,"to",up)

### Example values for testing

In [ ]:
J = np.array([[0,0,0,0],
              [0,0,0,0],
              [0,0,0,3],
              [0,0,3,0]])
Hon = np.array([1,2,3,4]) # local field value for each hysteron to switch + (on)
Hoff = np.array([-1,-2,-3,-4]) # ...and for each hysteron to switch - (off)

In [ ]:
# set initial conditions
S = [1.,1.,1.,1.]
H = -10.

In [ ]:
# execute to move up
H = u(Hon,J,S,H)
print(S)
print(H)

In [ ]:
# execute to move down
H = d(Hoff,J,S,H)
print(S)
print(H)

In [ ]:
# range of field values where current state is stable (if any)
stablerange(Hon,Hoff,J,S)

### T=2 graph in Fig. 6A

These thresholds and interactions produce a T=2 pathway. The parameters were discovered via a random search by Nathan C. Keim. 

In [ ]:
J = np.array([[ 0. , 0.9, 0.7, 0. ],
              [ 0.9, 0. ,-0.4, 0. ],
              [ 0.7,-0.4, 0. ,-0.7],
              [ 0. , 0. ,-0.7, 0. ]])
Hon  = np.array([ 1.,-0.1,-0.05, 0. ]) # local field value for each hysteron to switch + (on)
Hoff = np.array([ 0.,-0.3,-0.85,-0.85]) # ...and for each hysteron to switch - (off)

In [ ]:
# set initial conditions
S = [-1.,-1.,-1.,-1.]
H = -10.

In [ ]:
# execute to move up
H = u(Hon,J,S,H)
print(S)
print(H)

In [ ]:
# execute to move down
H = d(Hoff,J,S,H)
print(S)
print(H)

In [ ]:
S = [1.,1.,-1.,1.] # set state manually
stablerange(Hon,Hoff,J,S) # range of field values where state S is stable (if any)

### T=2 graph in Fig. 6B

The experiment navigates the downward avalanche via the unstable intermediate state $−+$$+−$, whereas the design in Fig. 6A passes through $+−$$+−$. We find that we can match the experiment by changing the bare threshold $\gamma^−_{1,0}$ (i.e., Hoff[0]) from $0$ to $0.85$. This can be tested below.

In [ ]:
J = np.array([[ 0. , 0.9, 0.7, 0. ],
              [ 0.9, 0. ,-0.4, 0. ],
              [ 0.7,-0.4, 0. ,-0.7],
              [ 0. , 0. ,-0.7, 0. ]])
Hon = np.array([ 1.,-0.1,-0.05, 0.  ]) # local field value for each hysteron to switch + (on)
Hoff = np.array([ 0.85,-0.3,-0.85,-0.85]) # ...and for each hysteron to switch - (off)

In [ ]:
# set initial conditions
S = [-1.,-1.,-1.,-1.]
H = -10.

In [ ]:
# execute to move up
H = u(Hon,J,S,H)
print(S)
print(H)

In [ ]:
# execute to move down
H = d(Hoff,J,S,H)
print(S)
print(H)

In [ ]:
S = [-1.,1.,1.,-1.] # set state manually
stablerange(Hon,Hoff,J,S) # range of field values where state S is stable (if any)